In [188]:
import pandas as pd                                                                     # type: ignore
import numpy as np                                                                      # type: ignore

### Importing dataset

In [189]:
import sys
sys.path.insert(1, "/home/mendrika/mendrika-phd/codes/nflics")
import nflics  

In [190]:
location = "Selibaby"
data = pd.read_csv(f"/home/mendrika/mendrika-phd/data/other-location/cleaned/train-data-{location}.csv", index_col=False)
test_data = pd.read_csv(f"/home/mendrika/mendrika-phd/data/other-location/cleaned/test-data-{location}.csv", index_col=False)

#### combining dataset and doing some exploratory analysis

In [191]:
data = data.dropna()
test_data = test_data.dropna()

In [192]:
dataset = "train"
if dataset == "train":
    data = data
else:
    data = test_data

In [193]:
data = data.drop([f"Cb_{location.lower()}_t2",	f"Cb_{location.lower()}_t3"], axis=1)

In [194]:
data['datetime'] = pd.to_datetime(data[['year', 'month', 'day', 'hour', 'minute']])

In [195]:
data = data.sort_values(by='datetime').reset_index(drop=True)

In [196]:
def find_exact_row_after_given_hours(row, hours, df):
    target_time = row['datetime'] + pd.Timedelta(hours=hours)
    corresponding_row = df[df['datetime'] == target_time]
    if not corresponding_row.empty:
        return corresponding_row.index[0]  # Return the index of the corresponding row
    else:
        return None

In [197]:
lead_time = 6
given_hours = lead_time - 1
data['corresponding_row_index'] = data.apply(find_exact_row_after_given_hours, args=(given_hours, data), axis=1)

In [199]:
data_specific_column = data[['corresponding_row_index']].copy()
data_specific_column[f'Cb_{location.lower()}_t{lead_time}'] = data_specific_column['corresponding_row_index'].apply(lambda x: data.loc[x, f'Cb_{location.lower()}_t1'] if pd.notna(x) else None)

data = data.drop([f"Cb_{location.lower()}_t1", "datetime",	"corresponding_row_index"], axis=1)

# Merge the specific column back to the original DataFrame
data = data.merge(data_specific_column[[f'Cb_{location.lower()}_t{lead_time}']], left_index=True, right_index=True)

In [200]:
data = data.dropna()

In [201]:
data.to_csv(f"/home/mendrika/mendrika-phd/data/other-location/cleaned/{dataset}-data-{location}-lt{lead_time}.csv", index=False)

In [13]:
def date_format(number):
    if number < 24:
        if len(str(number)) == 1:
            return f"0{number}"
        else:
            return str(number)
    else:
        return date_format(number - 24)

In [16]:
date_format(10)

'10'